# NDgpu — how much does a GPU buy on an *uncoupled* HP-MR transient?

No thermal coupling, no feedback: just the neutron-diffusion time loop on the
HP-MR microreactor core, so what is timed is the transient solver itself.

**The question is not "how fast is the GPU" but "which factor does it touch".**
A step is a fixed point over the end-of-step fission source; each sweep does a
few Gauss-Seidel passes over the G groups, and each pass solves one group system
with PCG:

    ms/step  =  (sweeps/step) x (G x subsweeps) x (CG iters/solve) x (cost/iter)

The first three factors are *algorithm*. They are set by the perturbation, the
tolerances and the accelerator, and they come out identical on both buses — §1
checks exactly that. Only `cost/iter` is a property of the machine, and a
speedup number is a claim about it alone. So every table below prints `cg/step`
next to the time; if that column moves between two legs, their wall-time ratio
is not a speedup.

**`us/cg` says what to optimize.** One CG iteration is a fixed number of kernels
over N-element arrays. Flat in problem size ⇒ launch-bound, and the lever is
fewer and larger launches. Rising with size ⇒ bandwidth-bound, and the lever is
fewer bytes. The crossover is where the GPU starts to pay, and it is a property
of this problem on this card — so it gets measured, not assumed.

The manoeuvre is a uniform +0.5 $ absorption step at t=0. It is shape-preserving
and near mesh-independent, so iteration counts stay comparable across the size
sweep (a drum rotation would not be: its absorber area fraction lands differently
on every mesh). Every timed step starts far from converged, which is the
expensive regime and the honest one.

In [ ]:
import os
try:                                        # Colab: upload dist/ndgpu-src.zip
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    get_ipython().run_line_magic("pip", f"install -q {zip_name}")
    try:
        import cupy
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q cupy-cuda12x")
    get_ipython().system("nvidia-smi -L")
except ImportError:                         # local run: ndgpu already importable
    pass

In [ ]:
import os

import numpy as np

from ndgpu import kernels
from ndgpu.benchmarks.hpmr_transient_bench import (HEADER, case_dof,
                                                   format_row, transient_bench)

try:
    import cupy
    HAVE_GPU = cupy.cuda.runtime.getDeviceCount() > 0
    print(cupy.cuda.runtime.getDeviceProperties(0)["name"].decode())
except Exception:
    HAVE_GPU = False
print("GPU available:", HAVE_GPU)

QUICK = bool(os.environ.get("NDGPU_QUICK"))

# 11-group ENDF/B-8 throughout: the 2-group placeholder set has a third of the
# group loop and none of the upscatter, so it would understate exactly the part
# of the work the GPU is being asked about.
BASE = dict(groups="11", steps=4, dt=0.02, anderson_depth=1, rebalance=True,
            max_sweeps=4000)

## 1. Both buses must do the same work

Before any timing. This is the gate the whole notebook rests on, and it is
also where the **batched group-source assembly** gets validated: on GPU the
per-group in-scatter loop collapses into one `kernels.group_accumulate` per row
over a dense `(G, G, *grid)` stack, a path that does not exist on CPU. If that
stack were built transposed the answer would still converge — to the wrong
number. So the power history is compared end to end, and the iteration counts
with it.

In [ ]:
cpu = transient_bench(refine=3, device="cpu", **BASE)
print(HEADER); print(format_row(cpu, "cpu 2d:3"))

if HAVE_GPU:
    gpu = transient_bench(refine=3, device="gpu", **BASE)
    print(format_row(gpu, "gpu 2d:3"))
    dP = abs(gpu["power"] - cpu["power"])
    dk = abs(gpu["k0"] - cpu["k0"])
    dcg = abs(gpu["cg_per_step"] - cpu["cg_per_step"]) / cpu["cg_per_step"]
    dsw = abs(gpu["sweeps_per_step"] - cpu["sweeps_per_step"]) / cpu["sweeps_per_step"]
    print(f"\n  |dP/P0| = {dP:.2e}   |dk0| = {dk:.2e}")
    print(f"  cg/step differs by {dcg:.3%}   sweeps/step by {dsw:.3%}")

    # THE tight invariant is the WORK, not the answer. Both buses run the same
    # algorithm on the same data and differ only in the ORDER of their
    # reductions, so the iteration counts must match almost exactly -- and this
    # is also the only one of the three checks that timing comparability
    # actually depends on. (Measured on a T4: 265 vs 265 sweeps, 56,665 vs
    # 56,664 CG.)
    assert dcg < 0.02 and dsw < 0.02, (
        "different iteration counts — the timings are not comparable")

    # The ANSWER is a much looser check, and deliberately so. A stopping
    # criterion is not an error bound. The step fixed point stops on the
    # relative CHANGE between successive iterates, and a tolerance ladder on
    # this problem (2D refine 2, tol_step 1e-5/-6/-7/-8) gives P(end) =
    # 1.625123 / 1.626469 / 1.626604 / 1.626618 — clean first-order convergence
    # to ~1.6266196, i.e. an error of about **150 x tol_step**. So at the
    # default 1e-6 the solver's own answer is uncertain at 1.5e-4, and the
    # measured CPU/GPU gap of 3.1e-5 is FIVE TIMES SMALLER than that: the two
    # buses agree better than the tolerance guarantees.
    #
    # The bound below therefore sits above the convergence floor but two orders
    # of magnitude under a real defect — a transposed scatter stack in the
    # batched assembly moves k by ~1e-3 and the power by percent.
    assert dk < 1e-6, f"k0 differs by {dk:.2e} — far beyond a convergence floor"
    assert dP < 1e-3, f"power differs by {dP:.2e} — far beyond a convergence floor"
    print("  -> same work, same answer; timings below are comparable")
    print(f"  (P itself is only converged to ~150 x tol_step = 1.5e-4 here, so"
          f" {dP:.1e} is well inside the solver's own error bar)")

### 1b. The batched assembly, switched off

The A/B that isolates it. `set_fused_group("groups", False)` sends the GPU back
down the per-`(g, g')` Python loop — two kernels and a full-size temporary per
coupling, i.e. O(G²) launches per subsweep instead of O(G). Same answer, more
launches.

In [ ]:
if HAVE_GPU:
    prev = kernels.set_fused_group("groups", False)
    unbatched = transient_bench(refine=3, device="gpu", **BASE)
    kernels.set_fused_group("groups", prev)
    print(HEADER)
    print(format_row(unbatched, "unbatched"))
    print(format_row(gpu, "batched"))
    print(f"\n  batched source assembly: {unbatched['ms_step']/gpu['ms_step']:.2f}x")
    print(f"  |dP| batched vs not: {abs(unbatched['power'] - gpu['power']):.2e}   "
          f"sweeps {unbatched['sweeps_per_step']:.0f} vs {gpu['sweeps_per_step']:.0f}")
    # Same reasoning as the gate above: these two also sum the group loops in
    # different orders, so they land at different iterates inside the same
    # convergence ball. The sweep count is the invariant that must hold tightly;
    # the power only has to stay far below what a wrong scatter stack would do.
    assert abs(unbatched["sweeps_per_step"] - gpu["sweeps_per_step"]) <= 1, (
        "batching changed the fixed point")
    assert abs(unbatched["power"] - gpu["power"]) < 3e-4, "batching changed the answer"

## 2. Speedup vs problem size

The headline table. The 2D core is small — a few thousand cells — and a GPU is
not expected to win there; the point of the sweep is to find *where* it starts
to, and the 3D extrusions are what carry the DOF count into the regime where it
should.

In [ ]:
# The CPU leg is skipped above CPU_DOF_MAX, set to INCLUDE 2d:6 (131k dof) --
# the most informative 2D point, since the measured GPU launch floor (~370 us/cg)
# puts break-even right around there -- and to EXCLUDE the 3D cases, where ~250
# outer sweeps per step would mean many hours on a Colab CPU. Budget ~1 h.
SIZES = ([(3, 0), (4, 0)] if QUICK else
         [(3, 0), (4, 0), (6, 0), (4, 10), (4, 20), (6, 20)])
CPU_DOF_MAX = 140_000

rows = []
print(HEADER)
for refine, nz in SIZES:
    label = f"2d:{refine}" if not nz else f"3d:{refine}x{nz}"
    c = g = None
    dof = case_dof(refine, nz)
    if dof <= CPU_DOF_MAX:
        c = transient_bench(refine, nz, device="cpu", **BASE)
        print(format_row(c, "cpu " + label), flush=True)
    else:
        print(f"{'cpu ' + label:>12}  {dof:>8,} dof -- skipped "
              f"(over the {CPU_DOF_MAX:,} dof budget)", flush=True)
    if HAVE_GPU:
        g = transient_bench(refine, nz, device="gpu", **BASE)
        print(format_row(g, "gpu " + label), flush=True)
    rows.append((label, c, g))

if HAVE_GPU:
    print(f"\n{'case':>12}  {'dof':>9}  {'cpu ms':>10}  {'gpu ms':>10}  "
          f"{'speedup':>8}  {'gpu us/cg':>10}  {'sweeps match':>13}")
    for label, c, g in rows:
        if c is None:
            print(f"{label:>12}  {g['dof']:>9,}  {'--':>10}  {g['ms_step']:>10.1f}  "
                  f"{'--':>8}  {g['us_per_cg']:>10.1f}  {'--':>13}")
            continue
        ok = abs(g["sweeps_per_step"] - c["sweeps_per_step"]) / c["sweeps_per_step"] < 0.02
        print(f"{label:>12}  {c['dof']:>9,}  {c['ms_step']:>10.1f}  "
              f"{g['ms_step']:>10.1f}  {c['ms_step']/g['ms_step']:>7.2f}x  "
              f"{g['us_per_cg']:>10.1f}  {'yes' if ok else 'NO':>13}")
    gus = [g["us_per_cg"] for _, _, g in rows if g]
    gdof = [g["dof"] for _, _, g in rows if g]
    print(f"\n  gpu us/cg spans {min(gus):.1f} to {max(gus):.1f} over "
          f"{min(gdof):,} to {max(gdof):,} dof "
          f"({max(gdof)/min(gdof):.0f}x the work).")
    print("  Roughly flat => launch-bound; roughly proportional => bandwidth-bound.")

## 3. The levers, one at a time

Each leg changes one thing against the §2 baseline at a fixed size. Two of them
are free (they only change how the existing kernels are dispatched); one changes
the arithmetic and has to be checked for accuracy, not just speed.

**`check_every`** — PCG's convergence test is a device→host reduction, i.e. a
full pipeline stall, and at the default of 1 there is one *per CG iteration*.
But this is a genuine trade, not a free win, and the measured `cg/g/sw` above
says how it should come out: the group solves run at a loose, sweep-adaptive
rtol and take only **~4–9 CG iterations each**, so skipping the test costs up to
`check_every - 1` wasted iterations on a solve that was already done — which on
a 5-iteration solve is a large fraction. Expect small values to win and large
ones to lose, and watch `cg/step` rise as it does: if `cg/step` climbs faster
than `ms/step` falls, the leg is a loss.

**`precond_degree`** — the Neumann polynomial preconditioner trades reductions
(syncs) for extra operator applies (one fused kernel each). On CPU this is a
straight loss — measured 37% slower on this problem — which is the expected
shape of a GPU win.

**float32** — halves the bytes moved. The tolerances here are absolute numbers
chosen for float64; if a leg fails to converge, that is the finding, and the fix
is tolerances that scale with the dtype rather than abandoning the lever.

In [ ]:
if HAVE_GPU:
    REF, NZ = (4, 0) if QUICK else (4, 20)
    base = transient_bench(REF, NZ, device="gpu", **BASE)
    legs = [("baseline", base)]

    for ce in (2, 3, 4):
        legs.append((f"check_every={ce}",
                     transient_bench(REF, NZ, device="gpu", check_every=ce, **BASE)))
    for pd in (1, 2):
        legs.append((f"precond_degree={pd}",
                     transient_bench(REF, NZ, device="gpu", precond_degree=pd, **BASE)))
    try:
        legs.append(("float32",
                     transient_bench(REF, NZ, device="gpu", dtype=np.float32, **BASE)))
    except Exception as e:
        print(f"float32 leg failed: {type(e).__name__}: {e}\n")

    print(f"{'leg':>20}  {'ms/step':>9}  {'vs base':>8}  {'cg/step':>8}  {'P(end)':>10}")
    for name, r in legs:
        print(f"{name:>20}  {r['ms_step']:>9.1f}  "
              f"{base['ms_step']/r['ms_step']:>7.2f}x  {r['cg_per_step']:>8.0f}  "
              f"{r['power']:>10.6f}")
    print("\n  P(end) must hold across every leg except float32, where a few "
          "units in the 6th figure is the dtype, not a bug.")

## 4. What is left on the table

**The largest lever is not the GPU, and this was settled before the GPU ran.**
The CPU baseline shows the within-step fixed point taking **~250 sweeps per
step** — 252 / 249 / 240 / 253 at 2D refine 2 / 3 / 4 / 6, i.e. flat in mesh
size, and on every step rather than just the first after the insertion. At
dt = 0.02 s the time term 1/(v·dt) ≈ 5e-8 is negligible against Σ_a ≈ 1e-2, so
backward Euler barely damps the source iteration and it converges at the core's
dominance ratio.

`rebalance=True` is what makes that tractable: it is a **one-cell CMFD** killing
the fundamental amplitude mode, and it is worth 5.3× (7,029 vs 36,959 ms/step at
refine 3, agreeing to 4e-4). What remains is the *shape* modes, and the
instrument for those is a **spatial** CMFD — generalizing the rebalance from one
coarse cell to a coarse mesh. `TransientSNSolver._cmfd_step` is a working one to
port from, worth 2.4–3.1× there on a fixed point the θ shift had *already*
damped. It is bus-independent and multiplies with whatever the GPU gives.

So read the tables above as optimizing the *second*-largest factor. What the
numbers decide:

- **`us/cg` flat across §2** ⇒ launch-bound, and the remaining kernel-level
  levers are structural: CUDA-graph capture of the CG iteration (fixed shapes,
  maximally repetitive — but `pcg` allocates and capture forbids that, so it
  needs a preallocated workspace), and batching the *group solves themselves*
  into one block system instead of G sequential ones, which trades Gauss-Seidel
  over groups for Jacobi and so buys launches at the cost of outer sweeps.
- **`us/cg` rising with dof** ⇒ bandwidth-bound, and float32 plus temporary
  elimination are the levers, not launch count. Note the CPU is *already* in
  this regime (ns/cg/dof settles at 4.2–4.8, ≈18 GB/s effective), so the
  asymptotic CPU→GPU ratio is just the bandwidth ratio.
- **`check_every`**: the cost side is already known — +19% CG iterations at
  `check_every=3`, sweeps and answer unchanged. The GPU's removed stalls have to
  beat that or the leg is negative.